[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/00_onboarding/00d_command_line_basics.ipynb)

# 🖥️ Notebook 0d — The command line (a survival guide)

> **Module:** Onboarding · **Estimated time:** 35–45 min · **Difficulty:** Absolute beginner · **Prerequisites:** none

Every other notebook in this course quietly assumes you can open a **terminal** and type a command — `pip install ...`, `python script.py`, `jupyter notebook`, `git commit`. If that sentence made you nervous, this lesson is for you. In 40 minutes you'll go from "what's a terminal?" to comfortably moving around, inspecting files, and chaining commands.

The command line looks intimidating because it's *quiet* — no buttons, no menus, just a blinking cursor. But it's the same handful of verbs over and over. Learn ~12 commands and you can operate any Unix machine (macOS, Linux, the cloud box your model will eventually run on, the CI runner in Module 14).

## 🎯 Learning objectives

By the end you can:

1. Explain what a **shell** is and read a command's *anatomy* (program, flags, arguments).
2. Move around the filesystem — `pwd`, `ls`, `cd` — and tell **absolute** from **relative** paths.
3. Create, read, move, copy and delete files and folders — `mkdir`, `touch`, `cat`, `cp`, `mv`, `rm`.
4. **Chain** commands with pipes `|` and redirect output with `>` — the superpower that makes the shell worth learning.
5. Understand `PATH`, environment variables, and why `python` sometimes "isn't found".
6. Run shell commands from inside a Jupyter notebook with `!` and `%%bash`.

> 🧭 **How this notebook stays runnable.** A real shell command depends on *where* you run it, which is terrible for a reproducible notebook. So every example below runs inside a **throwaway sandbox folder in `/tmp`** using Python's `subprocess` — you see the exact command and its real output, and nothing touches your actual files. The commands are ordinary bash; copy them into your own terminal and they work the same (minus the sandbox path).

## 1. What *is* the command line?

A **terminal** is a window that runs a **shell** — a program that reads a line you type, runs it, and prints the result. On macOS and Linux the default shell is usually **bash** or **zsh** (they're 95% identical for daily use); on Windows the closest equivalents are **Git Bash** or **WSL** (both give you the same Unix commands this lesson teaches).

A command has three parts. Read this the way the shell does, left to right:

```
   ls   -l   /tmp
   │    │     │
 program flag argument
 (what)  (how) (on what)
```

- **program** — the verb: `ls` = *list*.
- **flag(s)** — options that change behaviour, usually `-x` or `--word`. `-l` = *long format*.
- **argument(s)** — what to act on: the folder `/tmp`.

That's the whole grammar. Everything else is vocabulary.

Our helper below just runs a bash command and shows it with its output, like a real terminal session (`$` is the traditional prompt — you don't type it).

In [1]:
import subprocess, tempfile, os, textwrap

# One throwaway sandbox for the whole notebook — created fresh, deleted by the OS.
SANDBOX = tempfile.mkdtemp(prefix="cli_lesson_")

def sh(command, cwd=SANDBOX, show=True):
    """Run a bash command in the sandbox and print it like a terminal session."""
    result = subprocess.run(command, shell=True, cwd=cwd,
                            capture_output=True, text=True, executable="/bin/bash")
    if show:
        print(f"$ {command}")
        out = (result.stdout + result.stderr).rstrip()
        if out:
            print(out)
    return result

print("Sandbox ready at:", SANDBOX)
sh("echo Hello from the shell")

Sandbox ready at: /var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/cli_lesson_j5df1en5
$ echo Hello from the shell
Hello from the shell


CompletedProcess(args='echo Hello from the shell', returncode=0, stdout='Hello from the shell\n', stderr='')

## 2. Where am I? — `pwd`, and the filesystem tree

The filesystem is a **tree**: one root (`/`), folders inside folders, files at the leaves. At every moment your shell sits in one folder, the **current working directory** (cwd). Almost every "file not found" bug is really a "you're not where you think you are" bug.

- `pwd` — **p**rint **w**orking **d**irectory: *where am I right now?*

Think of the cwd as your cursor in a file explorer. Commands act relative to it unless you say otherwise.

In [2]:
sh("pwd")   # our sandbox is the current directory

$ pwd
/private/var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/cli_lesson_j5df1en5


CompletedProcess(args='pwd', returncode=0, stdout='/private/var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/cli_lesson_j5df1en5\n', stderr='')

## 3. Making and listing things — `mkdir`, `touch`, `ls`

- `mkdir NAME` — **m**a**k**e a **dir**ectory. `mkdir -p a/b/c` makes the whole chain at once (`-p` = *parents*).
- `touch FILE` — create an empty file (or update its timestamp).
- `ls` — **l**i**s**t the current directory. Useful flags:
  - `ls -l` — long format: permissions, size, modified time, name.
  - `ls -a` — **a**ll, including hidden files (names starting with `.`).
  - `ls -la` — both, combined (flags stack).

Let's build a tiny project and look at it.

In [3]:
sh("mkdir -p project/data")           # make project/ and project/data/ in one go
sh("touch project/app.py project/README.md project/data/sales.csv")
sh("touch project/.env")              # a hidden file (leading dot)
print()
sh("ls project")                      # plain list — note .env is hidden
print()
sh("ls -la project")                  # -a reveals .env; -l shows details

$ mkdir -p project/data
$ touch project/app.py project/README.md project/data/sales.csv
$ touch project/.env

$ ls project
README.md
app.py
data

$ ls -la project
total 0
drwxr-xr-x@ 6 christophweisser  staff  192 Jul 25 17:42 .
drwx------@ 3 christophweisser  staff   96 Jul 25 17:42 ..
-rw-r--r--@ 1 christophweisser  staff    0 Jul 25 17:42 .env
-rw-r--r--@ 1 christophweisser  staff    0 Jul 25 17:42 README.md
-rw-r--r--@ 1 christophweisser  staff    0 Jul 25 17:42 app.py
drwxr-xr-x@ 3 christophweisser  staff   96 Jul 25 17:42 data


CompletedProcess(args='ls -la project', returncode=0, stdout='total 0\ndrwxr-xr-x@ 6 christophweisser  staff  192 Jul 25 17:42 \x1b.\x1b[m\x1b[m\ndrwx------@ 3 christophweisser  staff   96 Jul 25 17:42 \x1b..\x1b[m\x1b[m\n-rw-r--r--@ 1 christophweisser  staff    0 Jul 25 17:42 .env\n-rw-r--r--@ 1 christophweisser  staff    0 Jul 25 17:42 README.md\n-rw-r--r--@ 1 christophweisser  staff    0 Jul 25 17:42 app.py\ndrwxr-xr-x@ 3 christophweisser  staff   96 Jul 25 17:42 \x1bdata\x1b[m\x1b[m\n', stderr='')

> 🔬 **Reading `ls -la`.** The first column (e.g. `-rw-r--r--`) is **permissions** (§8). The line starting with `d` is a directory (`data`). `.` is "this folder" and `..` is "the folder above" — you'll use `..` constantly to go up. Hidden files aren't secret; the leading dot just keeps config clutter out of the normal `ls`.

## 4. Moving around — `cd`, and absolute vs relative paths

`cd PATH` — **c**hange **d**irectory. This is the one command that changes *where you are*.

A **path** names a location two ways:

- **Absolute** — from the root: `/tmp/cli_lesson_x/project/data`. Starts with `/`. Unambiguous, works from anywhere.
- **Relative** — from the cwd: `project/data`, or `../..` to go up two levels. Shorter, but only meaningful given where you currently are.

Handy shorthands: `.` = here · `..` = one up · `~` = your home folder.

> ⚠️ **`cd` in a notebook is special.** Each `sh(...)` call is a *separate* shell that forgets everything when it ends — so a bare `cd` wouldn't "stick." That's actually how subprocesses work everywhere. To run a command *in* a folder we pass `cwd=...`; below, `sh(..., cwd=...)` plays the role of "having cd'd there first." In your own terminal a single `cd` persists for the whole session.

In [4]:
proj = os.path.join(SANDBOX, "project")

sh("pwd", cwd=proj)                    # same as: cd project; pwd
print()
sh("ls", cwd=proj)                     # relative: list project/
print()
sh("ls ..", cwd=proj)                  # .. = go up one → back to the sandbox
print()
sh(f"ls {SANDBOX}/project/data")       # absolute path works from anywhere

$ pwd
/private/var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/cli_lesson_j5df1en5/project

$ ls
README.md
app.py
data

$ ls ..
project

$ ls /var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/cli_lesson_j5df1en5/project/data
sales.csv


CompletedProcess(args='ls /var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/cli_lesson_j5df1en5/project/data', returncode=0, stdout='sales.csv\n', stderr='')

## 5. Looking inside files — `cat`, `echo`, and redirection `>`

- `echo TEXT` — print TEXT. Trivial alone, but the building block of scripts.
- `cat FILE` — dump a file's contents (con**cat**enate, historically).
- `>` — **redirect**: send a command's output *into a file* instead of the screen. `>` overwrites; `>>` appends.

Redirection is where the shell starts to feel powerful: any command that prints can write a file.

In [5]:
# Write two lines into a CSV using echo + redirection
sh('echo "date,amount" > project/data/sales.csv', )   # >  creates/overwrites
sh('echo "2026-01-01,120" >> project/data/sales.csv')  # >> appends
sh('echo "2026-01-02,90"  >> project/data/sales.csv')
print()
sh("cat project/data/sales.csv")                       # read it back

$ echo "date,amount" > project/data/sales.csv
$ echo "2026-01-01,120" >> project/data/sales.csv
$ echo "2026-01-02,90"  >> project/data/sales.csv

$ cat project/data/sales.csv
date,amount
2026-01-01,120
2026-01-02,90


CompletedProcess(args='cat project/data/sales.csv', returncode=0, stdout='date,amount\n2026-01-01,120\n2026-01-02,90\n', stderr='')

## 6. The superpower — pipes `|`

A **pipe** `|` feeds the output of one command straight into the *input* of the next, so you compose small tools into a bigger one. This is the Unix philosophy: many tiny programs that each do one thing, snapped together like Lego.

Three workhorses to pipe into:

- `wc -l` — count lines (**w**ord **c**ount, `-l` = lines).
- `grep PATTERN` — keep only lines matching PATTERN (your Ctrl-F for text streams).
- `sort` / `head` / `tail` — order, first-N, last-N.

Read `A | B | C` as "A, then feed that to B, then feed that to C."

In [6]:
# How many data rows (excluding the header) are in the CSV?
sh("cat project/data/sales.csv | grep 2026 | wc -l")   # keep 2026 lines, count them
print()
# List the project, keep only .py/.csv files, count them
sh("ls -1 project | grep -E '[.](py|csv)$'")
print()
# find: search the tree for files matching a pattern (great for big projects)
sh("find project -name '*.csv'")

$ cat project/data/sales.csv | grep 2026 | wc -l
       2

$ ls -1 project | grep -E '[.](py|csv)$'
app.py

$ find project -name '*.csv'
project/data/sales.csv


CompletedProcess(args="find project -name '*.csv'", returncode=0, stdout='project/data/sales.csv\n', stderr='')

> 🧠 **Why this matters for the rest of the course.** "How many rows in this log?", "which files import pandas?", "show me the last 20 lines of the training output" — all one-liners with pipes. You'll reach for `grep`/`wc`/`find` long before you write a Python script for the same thing.

## 7. Copy, move, rename, delete — `cp`, `mv`, `rm`

- `cp SRC DST` — **c**o**p**y. `cp -r` for a whole folder (`-r` = recursive).
- `mv SRC DST` — **m**o**v**e. Also how you **rename** (moving to a new name in the same folder).
- `rm FILE` — **r**e**m**ove. `rm -r FOLDER` for a folder.

> ⚠️ **`rm` does not use a Trash — it is instant and permanent.** There is no undo. Double-check the path before you press Enter, and be *extremely* wary of `rm -rf` (recursive + force): `rm -rf /` has ended careers. Treat every `rm` like a one-way door.

In [7]:
sh("cp project/README.md project/README_backup.md")    # copy
sh("mv project/app.py project/main.py")                # rename app.py → main.py
sh("ls project")
print()
sh("rm project/README_backup.md")                      # delete the copy
sh("ls project")

$ cp project/README.md project/README_backup.md
$ mv project/app.py project/main.py
$ ls project
README.md
README_backup.md
data
main.py

$ rm project/README_backup.md
$ ls project
README.md
data
main.py


CompletedProcess(args='ls project', returncode=0, stdout='README.md\n\x1bdata\x1b[m\x1b[m\nmain.py\n', stderr='')

## 8. Permissions and `PATH` — the two things beginners trip on

**Permissions.** That `-rw-r--r--` from `ls -l` is who-can-do-what: **r**ead, **w**rite, e**x**ecute, for *owner / group / everyone*. You mostly meet this as *"permission denied"* or *"command not found: ./script.sh"* — the fix is `chmod +x script.sh` (**ch**ange **mod**e, add e**x**ecute).

**`PATH`** is the single most useful environment variable. When you type `python`, the shell doesn't search your whole disk — it looks *only* in the folders listed in `PATH`, in order, and runs the first `python` it finds. *"command not found"* almost always means "that program's folder isn't on `PATH`" (or it isn't installed). `which python` tells you which one won.

In [8]:
# Make a script executable, then run it by path
sh('echo \'echo "I am a shell script"\' > project/run.sh')
sh("chmod +x project/run.sh")          # add execute permission
sh("ls -l project/run.sh")             # note the x's now in -rwxr-xr-x
sh("./run.sh", cwd=proj)               # run it (./ = "in this folder")
print()
# PATH: the ordered list of folders the shell searches for programs
sh("echo $PATH | tr ':' '\n' | head -5")   # split on ':' and show the first 5
print()
sh("which bash")                       # where does 'bash' actually live?

$ echo 'echo "I am a shell script"' > project/run.sh
$ chmod +x project/run.sh
$ ls -l project/run.sh
-rwxr-xr-x@ 1 christophweisser  staff  27 Jul 25 17:42 project/run.sh
$ ./run.sh
I am a shell script

$ echo $PATH | tr ':' '
' | head -5
/Users/christophweisser/Desktop/Coding/Python for AI-Driven Automation and Business Data Science/.venv/bin
/Users/christophweisser/.antigravity/antigravity/bin
/Users/christophweisser/.nvm/versions/node/v20.20.1/bin
/Users/christophweisser/Library/Application Support/Code/User/globalStorage/github.copilot-chat/debugCommand
/Users/christophweisser/Library/Application Support/Code/User/globalStorage/github.copilot-chat/copilotCli

$ which bash
/bin/bash


CompletedProcess(args='which bash', returncode=0, stdout='/bin/bash\n', stderr='')

> 🔬 **Environment variables** like `PATH` are `NAME=value` pairs the shell keeps in memory. Read one with `$NAME` (or `echo $NAME`), set one with `export NAME=value`. Module 8 uses exactly this to pass an API key (`export OPENAI_API_KEY=...`) without hard-coding it — a secret lives in the environment, never in your committed code.

## 9. The command line *inside* a notebook — `!` and `%%bash`

Jupyter lets you run shell commands without leaving the notebook — which is why you'll see `!pip install ...` all over this course.

- **`!command`** — run a single shell line. Put it right in a code cell.
- **`%%bash`** — make the *whole cell* a bash script (multiple lines).

(These are Jupyter conveniences, not Python. Under the hood they do what our `sh()` helper does. We show them as text here so this notebook stays runnable outside Jupyter too — try them for real in your own session.)

```python
!pwd                       # one line
!ls -la
!pip install pandas        # the exact idiom every setup cell uses
```

```bash
%%bash
for f in *.csv; do
    echo "found: $f"
done
```

**Rule of thumb:** `!` for one-offs (installs, a quick `ls`); real Python for anything with logic. Don't build a program out of `!` calls — that's what Python is for.

## 🧪 Practice exercises

Try each in your **own** terminal (or adapt the `sh(...)` helper). Solutions below.

### Exercise 1 — ⭐ Make a workspace
In one line each: make a folder `notes/`, create an empty `notes/todo.txt` inside it, and list `notes/` in long format.

### Exercise 2 — ⭐ Count matching lines
Given a file `log.txt`, print how many lines contain the word `ERROR`.

### Exercise 3 — ⭐⭐ Save a filtered list
Write, to a file `csvs.txt`, the names of every `.csv` file under a folder `project/` — one per line.

<details>
<summary>💡 <b>Solutions</b></summary>

```bash
# Exercise 1
mkdir notes
touch notes/todo.txt
ls -l notes

# Exercise 2  — grep -c counts matching lines directly
grep -c ERROR log.txt
# equivalently:  cat log.txt | grep ERROR | wc -l

# Exercise 3  — find prints matches; redirect the list into a file
find project -name '*.csv' > csvs.txt
cat csvs.txt
```

The pattern to internalise: **filter with `grep`/`find`, count with `wc -l`, capture with `>`.** Ninety percent of day-to-day shell work is some combination of those three.
</details>

## ✅ Self-assessment

- [ ] I can say what a shell is and name the three parts of a command.
- [ ] `pwd`, `ls -la`, `cd` — I can move around and read a long listing.
- [ ] I can tell an absolute path from a relative one, and use `.`, `..`, `~`.
- [ ] `mkdir`, `touch`, `cp`, `mv`, `rm` — and I respect that `rm` has no undo.
- [ ] I can pipe `grep`/`wc`/`find` together and redirect output with `>` / `>>`.
- [ ] I know what `PATH` is and why "command not found" happens.
- [ ] I can run a shell command from a notebook with `!`.

## 🧠 Key takeaways

1. The shell is a **verb + flags + arguments** grammar over a **tree** of folders — ~12 commands cover almost everything.
2. Most "file not found" errors are really **"wrong working directory."** Check `pwd` first.
3. **Pipes and redirection** (`|`, `>`) turn tiny commands into real tools — the reason the command line is worth learning.
4. `rm` is permanent, `PATH` explains "command not found", and `!`/`%%bash` bring all of this into your notebooks.

## 🚀 Next step

You can now operate a terminal — so you're ready to keep a *history* of your work. Continue with **[Notebook 0e — Git & GitHub](00e_git_and_github_basics.ipynb)**, which builds directly on the commands you just learned.

In [9]:
# Tidy up the sandbox (optional — the OS clears /tmp anyway)
import shutil
shutil.rmtree(SANDBOX, ignore_errors=True)
print("Sandbox removed. You reached the end — open Notebook 0e next.")

Sandbox removed. You reached the end — open Notebook 0e next.
